# Workflow 2 — Activity-selected (luminosity / fedd) kNN sensitivity

Steps:
1. Generate kNN summaries for activity-selected BHs
   (luminosity top-10%, fedd top-10%, luminosity top-2%)
2. Sensitivity + PCA for the luminosity top-10% selection
3. Selection-bias diagnosis: does the activity cut correlate with any
   CAMELS parameter? (uses `diagnose_selection_bias` against the full
   1000-sim parameter table, not just the retained subset)

Shared "load -> residual -> sensitivity" logic lives in
`src/analysis_pipeline.py`; the selection-bias check reuses
`src/selection_bias.py` instead of reimplementing the Spearman test.

In [1]:
import sys
sys.path.append("..")

import numpy as np

from src.generate_knn_active import run_suite
from src.analysis_pipeline import load_and_analyze, DEFAULT_PARAMS
from src.parameter_sensitivity import load_params
from src.selection_bias import diagnose_selection_bias

PARAMS_FILE = (
    "../CAMELS-master/docs/params/IllustrisTNG/"
    "CosmoAstroSeed_IllustrisTNG_L25n256_LH.txt"
)

## 1. Generate summaries

Set `GENERATE = False` once the .npz files already exist in `../outputs/`
to skip the (expensive) re-run.

In [2]:
GENERATE = True

SELECTIONS = [
    dict(snap=50, selection="luminosity", mass_cut=1e6, top_fraction=0.10),
    dict(snap=50, selection="fedd",       mass_cut=1e6, top_fraction=0.10),
]

if GENERATE:
    for kwargs in SELECTIONS:
        run_suite(**kwargs)
else:
    import os
    expected = [
        f"../outputs/knn_{s['selection']}_snap{s['snap']}_M{s['mass_cut']:.0e}.npz"
        for s in SELECTIONS
    ]
    missing = [f for f in expected if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(
            f"GENERATE=False but {len(missing)} expected file(s) are missing: {missing}"
        )

Found 1000 files


100%|██████████| 1000/1000 [03:21<00:00,  4.96it/s]


Saved:
../outputs/knn_luminosity_snap50_M1e+06.npz

Successful runs = 1000
Summary shape = (1000, 150)
Mean N_BH = 114.881
Found 1000 files



100%|██████████| 1000/1000 [03:16<00:00,  5.09it/s]


Saved:
../outputs/knn_fedd_snap50_M1e+06.npz

Successful runs = 1000
Summary shape = (1000, 150)
Mean N_BH = 114.881


## 2. Sensitivity + PCA — luminosity top-10%

In [3]:
result = load_and_analyze(
    "../outputs/knn_luminosity_snap50_M1e+06.npz",
    params=DEFAULT_PARAMS,
    params_file=PARAMS_FILE,
    run_pca_flag=True,
)

sim_ids   = result["sim_ids"]
residuals = result["residuals"]
theta     = result["theta"]          # params for the *retained* sims only

print("PC1 =", result["explained"][0])
print(result["sensitivity"].to_string(index=False))

PC1 = 0.5010022599720857
parameter      rms
  Omega_m 0.008387
  sigma_8 0.002655
    A_SN1 0.003374
   A_AGN1 0.001815
    A_SN2 0.002351
   A_AGN2 0.001216


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


## 3. Selection-bias diagnosis

Tests whether the luminosity-top-10% cut preferentially retains
simulations from particular regions of parameter space. Unlike a
naive Spearman test against the *retained* theta only, this loads
the full 1000-sim parameter table (`theta_all`) and compares
retained vs. full -- which is what `diagnose_selection_bias` needs
to actually mean anything.

In [4]:
all_ids = np.arange(1000)

theta_all = load_params(all_ids, PARAMS_FILE)

diag_df = diagnose_selection_bias(
    retained_ids=sim_ids,
    all_ids=all_ids,
    theta_all=theta_all,
    params=DEFAULT_PARAMS,
)

retained_fraction = len(sim_ids) / len(all_ids)

print("=" * 60)
print("SELECTION FUNCTION TEST")
print("=" * 60)
print(diag_df.to_string(index=False))
print()
print("Retained fraction =", retained_fraction)

if diag_df["bias_flag"].any():
    flagged = diag_df.loc[diag_df["bias_flag"], "parameter"].tolist()
    print()
    print(f"WARNING: possible selection bias in: {flagged}")
    print("Consider using compute_ipw_weights() / weighted_sensitivity_table()")
    print("from src.selection_bias to correct the sensitivity analysis.")

SELECTION FUNCTION TEST
parameter  spearman_rho  spearman_p  ks_stat  ks_p  retained_mean  full_mean  bias_flag
  Omega_m           0.0         1.0      0.0   1.0       0.300000   0.300000      False
  sigma_8           0.0         1.0      0.0   1.0       0.800000   0.800000      False
    A_SN1           0.0         1.0      0.0   1.0       1.352526   1.352526      False
   A_AGN1           0.0         1.0      0.0   1.0       1.352526   1.352526      False
    A_SN2           0.0         1.0      0.0   1.0       1.082021   1.082021      False
   A_AGN2           0.0         1.0      0.0   1.0       1.082021   1.082021      False

Retained fraction = 1.0


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


In [5]:
from src.selection_bias import diagnose_selection_bias, diagnose_activity_cut_bias  # add this import

# ... existing diagnose_selection_bias(...) block stays as-is ...

# NEW: does the activity cut itself correlate with any CAMELS parameter?
# (diagnose_selection_bias only catches whole-sim dropout, which is 0%
#  here — this catches whether the *size* of the retained top-10% BH
#  sample varies systematically with theta.)
activity_diag_df = diagnose_activity_cut_bias(
    sim_ids=sim_ids,
    nbh=result["nbh"],
    theta_all=theta_all,
    params=DEFAULT_PARAMS,
)

print()
print("=" * 60)
print("ACTIVITY-CUT SIZE BIAS TEST")
print("=" * 60)
print(activity_diag_df.to_string(index=False))

if activity_diag_df["bias_flag"].any():
    flagged = activity_diag_df.loc[activity_diag_df["bias_flag"], "parameter"].tolist()
    print()
    print(f"WARNING: retained sample size correlates with: {flagged}")


ACTIVITY-CUT SIZE BIAS TEST
parameter  spearman_rho  spearman_p  bias_flag
  Omega_m      0.980311    0.000000       True
  sigma_8      0.143655    0.000005       True
    A_SN1     -0.078378    0.013166      False
   A_AGN1      0.003234    0.918642      False
    A_SN2      0.018618    0.556496      False
   A_AGN2     -0.007503    0.812674      False

